# Spring Validation — Parameterized Template

Run validation workloads for any spring through the NUCLEUS composition.
Select a spring, dispatch via ToadStool, parse results, visualize.

**Tier 1**: Parse structured CLI output from validation binaries
**Tier 2** (evolution target): Direct JSON-RPC to primal APIs

In [ ]:
import subprocess, json, re, os
from pathlib import Path
from collections import defaultdict

# Primal ports (Phase 59 canonical)
PRIMALS = {
    'beardog': 9100, 'songbird': 9200, 'squirrel': 9300,
    'toadstool': 9400, 'nestgate': 9500, 'rhizocrypt': 9602,
    'loamspine': 9700, 'coralreef': 9730, 'barracuda': 9740,
    'skunkbat': 9140, 'biomeos': 9800, 'sweetgrass': 9850,
    'petaltongue': 9900
}

NUCLEUS_ROOT = Path(os.environ.get('NUCLEUS_PROJECT_ROOT', Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent))
WORKLOADS_DIR = NUCLEUS_ROOT / 'workloads'
TOADSTOOL_BIN = os.environ.get('TOADSTOOL_BIN', 'toadstool')

## 1. Health Check — All 13 Primals

In [ ]:
import socket

def check_primal(name, port, timeout=2):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        s.connect(('127.0.0.1', port))
        s.close()
        return True
    except (socket.timeout, ConnectionRefusedError, OSError):
        return False

print('NUCLEUS Health Check')
print('=' * 50)
alive = 0
for name, port in sorted(PRIMALS.items(), key=lambda x: x[1]):
    ok = check_primal(name, port)
    status = 'ALIVE' if ok else 'DOWN'
    alive += ok
    print(f'  {name:15s}  :{port}  {status}')
print(f'\n{alive}/{len(PRIMALS)} primals healthy')

## 2. Select Spring

In [ ]:
# Available springs with workload TOMLs
available = {}
for d in sorted(WORKLOADS_DIR.iterdir()):
    if d.is_dir() and d.name != 'templates':
        tomls = list(d.glob('*.toml'))
        if tomls:
            available[d.name] = tomls

print('Available springs with workload TOMLs:')
for spring, tomls in available.items():
    print(f'  {spring}: {len(tomls)} workload(s)')
    for t in tomls:
        print(f'    - {t.name}')

# Select spring (change this to run a different spring)
SPRING = 'wetspring'
print(f'\nSelected: {SPRING}')

## 3. Dispatch Validation via ToadStool

In [ ]:
def dispatch_workload(toml_path, timeout=120):
    """Execute a workload TOML through ToadStool and capture output."""
    try:
        result = subprocess.run(
            [TOADSTOOL_BIN, 'execute', str(toml_path)],
            capture_output=True, text=True, timeout=timeout
        )
        return {
            'toml': toml_path.name,
            'returncode': result.returncode,
            'stdout': result.stdout,
            'stderr': result.stderr
        }
    except subprocess.TimeoutExpired:
        return {'toml': toml_path.name, 'returncode': -1, 'stdout': '', 'stderr': 'TIMEOUT'}
    except FileNotFoundError:
        return {'toml': toml_path.name, 'returncode': -1, 'stdout': '', 'stderr': f'{TOADSTOOL_BIN} not found'}

if SPRING in available:
    results = []
    for toml in available[SPRING]:
        print(f'Dispatching: {toml.name} ...', end=' ')
        r = dispatch_workload(toml)
        results.append(r)
        print(f'exit={r["returncode"]}')
else:
    print(f'No workloads found for {SPRING}')
    results = []

## 4. Parse Structured Output

In [ ]:
def parse_checks(stdout):
    """Parse [OK]/[FAIL] structured output into sections and checks."""
    sections = defaultdict(list)
    current_section = 'General'
    ok_pattern = re.compile(r'\[OK\]\s+(.*)')
    fail_pattern = re.compile(r'\[FAIL\]\s+(.*)')
    section_pattern = re.compile(r'^\s*(?:──\s*)?([A-Z][A-Za-z0-9 /&_-]+)(?:\s*──)?\s*$')
    summary_pattern = re.compile(r'(\d+)/(\d+) checks passed')

    for line in stdout.split('\n'):
        m = section_pattern.match(line.strip())
        if m and not ok_pattern.search(line) and not fail_pattern.search(line):
            current_section = m.group(1).strip()
            continue
        m = ok_pattern.search(line)
        if m:
            sections[current_section].append(('OK', m.group(1).strip()))
            continue
        m = fail_pattern.search(line)
        if m:
            sections[current_section].append(('FAIL', m.group(1).strip()))

    m = summary_pattern.search(stdout)
    summary = (int(m.group(1)), int(m.group(2))) if m else None
    return dict(sections), summary

all_checks = []
for r in results:
    sections, summary = parse_checks(r['stdout'])
    total_ok = sum(1 for checks in sections.values() for s, _ in checks if s == 'OK')
    total_fail = sum(1 for checks in sections.values() for s, _ in checks if s == 'FAIL')
    print(f"{r['toml']}: {total_ok} OK, {total_fail} FAIL ({len(sections)} sections)")
    all_checks.append({'workload': r['toml'], 'sections': sections, 'summary': summary,
                       'ok': total_ok, 'fail': total_fail})

## 5. Visualize Results

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if all_checks:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Per-workload bar chart
    names = [c['workload'].replace('.toml', '').replace(f'{SPRING}-', '') for c in all_checks]
    oks = [c['ok'] for c in all_checks]
    fails = [c['fail'] for c in all_checks]
    x = range(len(names))
    ax1.barh(x, oks, color='#2ecc71', label='OK')
    ax1.barh(x, fails, left=oks, color='#e74c3c', label='FAIL')
    ax1.set_yticks(x)
    ax1.set_yticklabels(names, fontsize=8)
    ax1.set_xlabel('Checks')
    ax1.set_title(f'{SPRING} — Per Workload')
    ax1.legend()

    # Aggregate pie
    total_ok = sum(oks)
    total_fail = sum(fails)
    ax2.pie([total_ok, total_fail] if total_fail else [total_ok],
            labels=['OK', 'FAIL'] if total_fail else ['OK'],
            colors=['#2ecc71', '#e74c3c'] if total_fail else ['#2ecc71'],
            autopct='%1.0f%%', startangle=90)
    ax2.set_title(f'{SPRING} — {total_ok}/{total_ok + total_fail} checks')

    plt.tight_layout()
    plt.savefig(f'/tmp/{SPRING}_validation.png', dpi=150)
    plt.show()
    print(f'Saved: /tmp/{SPRING}_validation.png')
else:
    print('No results to visualize')

## 6. Provenance Inspection

In [ ]:
def rpc_call(port, method, params=None):
    """Send a JSON-RPC 2.0 request to a primal."""
    payload = json.dumps({
        'jsonrpc': '2.0', 'method': method,
        'params': params or {}, 'id': 1
    }) + '\n'
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(5)
        s.connect(('127.0.0.1', port))
        s.sendall(payload.encode())
        data = b''
        while True:
            chunk = s.recv(4096)
            if not chunk:
                break
            data += chunk
            if b'\n' in data:
                break
        s.close()
        return json.loads(data.decode().strip())
    except Exception as e:
        return {'error': str(e)}

# Query rhizoCrypt for recent DAG sessions
print('Querying rhizoCrypt (DAG provenance)...')
resp = rpc_call(PRIMALS['rhizocrypt'], 'dag.list_sessions')
if 'result' in resp:
    sessions = resp['result']
    print(f'  {len(sessions)} DAG session(s)')
    if sessions:
        latest = sessions[-1]
        print(f'  Latest: {latest.get("id", "?")}')
        print(f'  Events: {latest.get("event_count", "?")}')
else:
    print(f'  Error: {resp.get("error", "unknown")}')

# Query loamSpine for ledger state
print('\nQuerying loamSpine (permanent ledger)...')
resp = rpc_call(PRIMALS['loamspine'], 'spine.status')
if 'result' in resp:
    print(f'  Ledger entries: {resp["result"].get("entry_count", "?")}')
else:
    print(f'  Error: {resp.get("error", "unknown")}')

## 7. Tier 2 Stubs — Direct JSON-RPC to Primal APIs

When primals evolve to expose validation methods via JSON-RPC,
notebooks can call them directly instead of parsing CLI output.
These stubs show the target API signatures.

In [ ]:
# === TIER 2 — Evolution Target (not yet implemented in primals) ===
#
# When toadstool.validate is available:
#
# resp = rpc_call(PRIMALS['toadstool'], 'toadstool.validate', {
#     'workload': 'wetspring-16s-rust-validation',
#     'format': 'json'
# })
# checks = resp['result']['checks']  # structured check results
# df = pd.DataFrame(checks)
#
# When toadstool.list_workloads is available:
#
# resp = rpc_call(PRIMALS['toadstool'], 'toadstool.list_workloads', {
#     'spring': 'wetspring'
# })
# workloads = resp['result']  # list of available workload names
#
# When barracuda.compute is available:
#
# resp = rpc_call(PRIMALS['barracuda'], 'barracuda.compute', {
#     'shader': 'shannon_entropy_f64',
#     'input': [1.0, 2.0, 3.0, 4.0]
# })
# result = resp['result']['output']  # GPU compute result
#
# When biomeos.spring_status is available:
#
# resp = rpc_call(PRIMALS['biomeos'], 'biomeos.spring_status')
# springs = resp['result']  # which springs have binaries available
#
# When nestgate.artifact_query is available:
#
# resp = rpc_call(PRIMALS['nestgate'], 'nestgate.artifact_query', {
#     'hash': 'b106aa1d1bb45430d00d605626e10488119f9e4f9f315a738939049a6da9ceec'
# })
# provenance = resp['result']  # artifact provenance chain

print('Tier 2 stubs defined — see LIVE_SCIENCE_API.md for the evolution spec')